# Separação do Dataset Curado (Train, Val, Test)

Este notebook executa o split final do dataset consolidado. Ele lê as imagens sobreviventes que estão na pasta `data/visual/` (as imagens que você escolheu manter) e as divide de forma **estratificada** em conjuntos de treino, validação e teste para o treinamento do YOLO.

In [ ]:
# Configura o path do projeto
import sys
import os
sys.path.append(os.path.abspath('..'))

## 1. Executar a Separação (Split)

Abaixo definimos as proporções de separação (`70% Treino / 20% Validação / 10% Teste` por padrão) e executamos o script `src/data/split.py`.

In [ ]:
from src.data.split import run_split

# Executa a separação (base_dir='..' indica que a pasta raiz do projeto está um nível acima)
run_split(train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42, base_dir='..')

## 2. Análise Visual do Dataset Final

Vamos plotar gráficos para visualizar como os dados foram divididos e comprovar a distribuição estratificada.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carrega a rastreabilidade final
df_final = pd.read_csv('../data/metadata/traceability_final.csv')

# Configura estilo visual dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Tamanho de cada split
split_counts = df_final['split'].value_counts().reindex(['train', 'val', 'test'])
colors = ['#1f77b4', '#aec7e8', '#ff7f0e']
split_counts.plot(kind='bar', color=colors, ax=ax1, edgecolor='black', alpha=0.85)
ax1.set_title('Número de Imagens por Conjunto (Split)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Conjunto (Split)', fontsize=12)
ax1.set_ylabel('Quantidade de Imagens', fontsize=12)
ax1.tick_params(axis='x', rotation=0)
for i, val in enumerate(split_counts):
    ax1.text(i, val + (split_counts.max() * 0.01), str(val), ha='center', fontweight='bold')

# Gráfico 2: Distribuição dos datasets de origem dentro de cada split (Estratificação)
pivot = df_final.groupby(['original_dataset', 'split']).size().unstack(fill_value=0)
pivot = pivot[['train', 'val', 'test']]
pivot.plot(kind='bar', stacked=True, ax=ax2, colormap='tab20', edgecolor='black', alpha=0.85)
ax2.set_title('Origem das Imagens por Dataset (Estratificado)', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Dataset de Origem', fontsize=12)
ax2.set_ylabel('Quantidade de Imagens', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Conjunto (Split)', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

### Detalhamento em Tabela

Esta tabela mostra exatamente a quantidade de imagens enviadas para cada split por base de dados de origem.

In [ ]:
pivot_table = df_final.pivot_table(index='original_dataset', columns='split', aggfunc='size', fill_value=0)
pivot_table = pivot_table[['train', 'val', 'test']]
pivot_table['Total'] = pivot_table.sum(axis=1)

# Porcentagens aproximadas
for split in ['train', 'val', 'test']:
    pivot_table[f'{split}_%'] = (pivot_table[split] / pivot_table['Total'] * 100).round(1)
    
display(pivot_table[['train', 'train_%', 'val', 'val_%', 'test', 'test_%', 'Total']])